In [ ]:
import json, re
import requests

import numpy as np

from pathlib import Path
from urllib.parse import urlencode

# First, get 'teams_full' array: [<dict1>, <dict2>], with <dict#> = { <pokename> : <dict_of_info> }
# ===========================
REPLAY_DIR = Path('./../data/test_data_replays/')

with open('./../data/POKEDEX.json', 'r') as file:
    POKEDEX = json.load(file)
with open('./../data/POKEDEX_for_test.json', 'r') as file:
    POKEDEX_for_test = json.load(file)
with open('./pokedex_raw.json', 'r') as file:
    POKEDEX_raw = json.load(file)

from battle import *
from full_pokemon import FullPokemon

def pull_by_num(num, ns='', parse=False):
    return Battle(f'./../data/test_data_replays/gen9randombattle-{num}.json', parse)

In [11]:
POKEDEX = { item['id'] : {key:item.get(key) for key in item.keys()} for item in POKEDEX_raw }
with open('./../data/POKEDEX_for_test.json', 'w') as file:
    json.dump(POKEDEX, file)

In [3]:
d = REPLAY_DIR.iterdir()
L = [ next(d) for _ in range(500)]
for file in L : 
    if file.is_file() and file.name.endswith('.json') : 
        bat = Battle(file)
    elif file.name.startswith('.') :
        continue
    else: 
        print(f"made it to {file.name}")
        break

In [19]:
errs = []

for replay in REPLAY_DIR.iterdir(): 
    if replay.is_file() and replay.name.endswith('.json'): # skipping hidden files
        try:
            # bat = Battle(file, parse=True)
            with replay.open() as file:
                battle_json = json.load(file)

            battle_json['player_dets'] = get_player_dets(battle_json)
            
            TEAMS = get_teams_full(battle_json)
            try :
                for team in TEAMS:
                    append_team_stats(team, battle_json['id'])
            except : 
                errs.append(replay.name)
                continue
            battle_json['teams_full'] = TEAMS
            
            replay.write_text(json.dumps(battle_json), encoding='utf-8')
        
        except:
            print("Error parsing file: %s" % replay.name)
            errs.append(replay.name)
            continue
    else:
        print("Could not open file: %s" % replay.name)
        errs.append(replay.name)
        continue